[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camiloandcu/Retail-Demand-Forecasting/blob/main/02_feature_pipeline.ipynb)

# 02 - Leakage-safe features and rolling-origin validation

**Purpose.** Demonstrate that every model input is available at forecast creation time and that validation reproduces the complete Kaggle horizon. No model is trained.

**Inputs:** `configs/full.yaml`, the seven validated Kaggle CSVs, and `retail_forecast`. Set `RETAIL_FORECAST_MODE=smoke` for deterministic schema-only fixtures.

**Outputs:** `artifacts/metrics/split_manifest.json` and `artifacts/metrics/feature_manifest.json`. The transformed panel is intentionally not exported.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/camiloandcu/Retail-Demand-Forecasting.git"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    REPO_ROOT = Path("/content/Retail-Demand-Forecasting")
    if not (REPO_ROOT / "pyproject.toml").is_file():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        subprocess.run(
            ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", "main"],
            check=True,
        )
else:
    REPO_ROOT = Path.cwd().resolve()
    if not (REPO_ROOT / "pyproject.toml").is_file():
        raise RuntimeError("Run the notebook from the repository root.")
os.chdir(REPO_ROOT)
if importlib.util.find_spec("pip") is not None:
    install_command = [sys.executable, "-m", "pip", "install", "-e", ".[notebook,dev]"]
elif shutil.which("uv"):
    install_command = ["uv", "pip", "install", "--python", sys.executable, "-e", ".[notebook,dev]"]
else:
    raise RuntimeError("This environment provides neither pip nor uv.")
subprocess.run(install_command, check=True)
source_path = str(REPO_ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)
print(f"Repository ready at {REPO_ROOT} with Python {sys.version.split()[0]}")

In [ ]:
import resource
import time

import pandas as pd
from IPython.display import display

from retail_forecast.acquisition import download_competition_data
from retail_forecast.config import PROJECT_HORIZON, load_config
from retail_forecast.data import (
    EXPECTED_KAGGLE_SHA256,
    load_frames,
    required_files_exist,
    validate_dataset,
    write_synthetic_dataset,
)
from retail_forecast.eda import validate_kaggle_hashes
from retail_forecast.features import (
    FeatureSpec,
    build_fold_feature_batches,
    export_feature_manifest,
    feature_manifest,
    fit_preprocessor,
    model_input_contract,
    transform_features,
)
from retail_forecast.splits import (
    export_split_manifest,
    make_rolling_origin_folds,
    split_manifest,
)
from retail_forecast.versions import capture_versions

RUN_STARTED = time.perf_counter()
MODE = os.getenv("RETAIL_FORECAST_MODE", "full").strip().lower()
if MODE not in {"full", "smoke"}:
    raise ValueError("RETAIL_FORECAST_MODE must be 'full' or 'smoke'.")
config = load_config(f"configs/{MODE}.yaml")
spec = FeatureSpec(horizon=config.forecast.horizon)
print(f"mode={MODE} | horizon={config.forecast.horizon} | feature_spec={spec.sha256[:12]}")

## Acquire and validate inputs

Full mode follows the same secure acquisition contract as Notebook 01. In Colab, add `KAGGLE_API_TOKEN` under **Secrets** and grant notebook access. The value is read only when files are absent, is never printed or written to Git, and is removed from the environment after download.

In [ ]:
if config.data.source == "synthetic":
    if not required_files_exist(config.paths.raw_data_dir):
        write_synthetic_dataset(config)
elif not required_files_exist(config.paths.raw_data_dir):
    token_loaded = False
    if IN_COLAB:
        from google.colab import userdata

        kaggle_token = userdata.get("KAGGLE_API_TOKEN")
        if not kaggle_token:
            raise RuntimeError("Add KAGGLE_API_TOKEN to Colab Secrets and grant access.")
        os.environ["KAGGLE_API_TOKEN"] = kaggle_token
        token_loaded = True
    try:
        download_competition_data(config.paths.raw_data_dir)
    finally:
        if token_loaded:
            os.environ.pop("KAGGLE_API_TOKEN", None)
            del kaggle_token
if MODE == "full":
    validate_kaggle_hashes(config.paths.raw_data_dir)
dataset_summary = validate_dataset(config)
frames = load_frames(config.paths.raw_data_dir)
display(dataset_summary.to_dict())

## Series, origin, and row contract

A series is identified by `(store_nbr, family)` and ordered by daily `date`. One direct-model row is keyed by `(store_nbr, family, forecast_origin, target_date, horizon)`. The 16 horizons correspond to the 16 dates present in `test.csv`.

Historical features are anchored at `forecast_origin`, not shifted from each target date. Consequently, all 16 rows from one origin share the same observed sales/transaction history; steps 2–16 cannot read earlier targets from their own forecast block.

In [ ]:
folds = make_rolling_origin_folds(config, frames["train.csv"]["date"])
split_data = split_manifest(config, folds)
METRICS_DIR = REPO_ROOT / "artifacts/metrics"
split_path = export_split_manifest(config, folds, METRICS_DIR / "split_manifest.json")
display(pd.DataFrame([fold.to_dict() for fold in folds]))
print(f"split manifest: {split_path} | sha256={split_data['sha256']}")

## Feature availability policy

Sales and transactions stop at the origin. Sales lags use 1, 7, 14, 28, and 56 origin-relative days; rolling statistics use windows of 7, 14, 28, and 56 days and always end at the origin. Transactions use origin-relative lags and windows only.

Target-date features are limited to deterministic calendar values and covariates supplied by Kaggle for that date: promotions, oil, and resolved holidays. Holiday locale is mapped through store city/state and aggregated to a unique store-date row before joining.

In [ ]:
fold_runs = []
example_batch = None
example_fitted = None
example_transformed = None
for fold in folds:
    started = time.perf_counter()
    training_batch, validation_batch = build_fold_feature_batches(frames, fold, spec)
    fitted = fit_preprocessor(training_batch, fold.origin)
    transformed_train = transform_features(training_batch, fitted)
    transformed_validation = transform_features(validation_batch, fitted)
    elapsed = time.perf_counter() - started
    raw_bytes = int(
        training_batch.frame.memory_usage(deep=True).sum()
        + validation_batch.frame.memory_usage(deep=True).sum()
    )
    transformed_bytes = int(
        transformed_train.memory_usage(deep=True).sum()
        + transformed_validation.memory_usage(deep=True).sum()
    )
    fold_runs.append(
        {
            "fold": fold.name,
            "origin": fold.origin.date().isoformat(),
            "training_feature_origin": training_batch.origin.date().isoformat(),
            "preprocessor_fitted_through": fitted.fitted_through.date().isoformat(),
            "training_rows": len(training_batch.frame),
            "validation_rows": len(validation_batch.frame),
            "feature_count": validation_batch.feature_count,
            "raw_feature_bytes": raw_bytes,
            "transformed_bytes": transformed_bytes,
            "elapsed_seconds": round(elapsed, 4),
            "process_max_rss_kib": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
            "preprocessor_sha256": fitted.sha256,
            "validation_dates": validation_batch.frame["target_date"].nunique(),
            "validation_series": validation_batch.frame[["store_nbr", "family"]]
            .drop_duplicates()
            .shape[0],
            "duplicate_row_keys": int(
                validation_batch.frame.duplicated(
                    ["store_nbr", "family", "forecast_origin", "target_date", "horizon"]
                ).sum()
            ),
            "missing_before_transform": int(validation_batch.frame.isna().sum().sum()),
            "missing_after_transform": int(transformed_validation.isna().sum().sum()),
            "history_missing_rates": {
                column: round(float(validation_batch.frame[column].mean()), 6)
                for column in validation_batch.numeric_features
                if column.startswith(("sales_", "transactions_")) and column.endswith("_missing")
            },
            "known_future_missing_rates": {
                "onpromotion": round(float(validation_batch.frame["onpromotion"].isna().mean()), 6),
                "oil_target": round(float(validation_batch.frame["oil_target"].isna().mean()), 6),
            },
            "unknown_category_rates": {
                column: round(float(transformed_validation[f"cat__{column}"].eq(0).mean()), 6)
                for column in validation_batch.categorical_features
            },
            "vocabulary_sizes": {
                key: len(values) for key, values in fitted.categorical_vocabulary.items()
            },
        }
    )
    example_batch = validation_batch
    example_fitted = fitted
    example_transformed = transformed_validation
display(pd.DataFrame(fold_runs))

## Imputation and categorical fitting

Numeric missing values use medians learned only from the preceding supervised training block. A completely missing train column falls back to zero while its missing/count indicator remains available. There is no backward fill or bidirectional interpolation.

Categorical vocabularies are also train-only. Code 0 is reserved for `__UNKNOWN__`, so a store, family, or metadata value first encountered in validation cannot expand the fitted vocabulary.

In [ ]:
assert example_batch is not None
assert example_fitted is not None
assert example_transformed is not None
preprocessing_evidence = pd.DataFrame(
    {
        "value": [
            example_batch.feature_count,
            len(example_batch.numeric_features),
            len(example_batch.categorical_features),
            int(example_batch.frame.isna().sum().sum()),
            int(example_transformed.isna().sum().sum()),
            example_fitted.fitted_through.date().isoformat(),
        ]
    },
    index=[
        "features",
        "numeric features",
        "categorical features",
        "missing before fold-fit transform",
        "missing after fold-fit transform",
        "preprocessor fitted through",
    ],
)
display(preprocessing_evidence)

## Negative leakage tests

The tests below must reject a 15-day regression, future sales or transaction history, train/validation overlap, preprocessing fitted beyond its cutoff, and unsafe holiday joins. A sentinel test replaces every validation target with an extreme value and verifies that the feature matrix does not change.

In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_splits.py", "tests/test_features.py"],
    check=False,
    capture_output=True,
    text=True,
)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr, file=sys.stderr)
if test_result.returncode != 0:
    raise RuntimeError("Leakage tests failed; model development is blocked.")
print("Leakage and split tests: PASS")

## Contracts handed to later models

The tabular contract is one encoded row per series–origin–horizon. Numeric values are `float32`; categorical codes are `int32`; identifiers and dates remain available for audit.

The recurrent contract keeps history ending at the origin, known-future covariates shaped across 16 steps, static categories for embeddings, and a target shaped `[batch, 16]`. This notebook defines that boundary but does not implement an LSTM or GRU.

In [ ]:
assert example_batch is not None
assert example_transformed is not None
gate_checks = {
    "real horizon is 16": PROJECT_HORIZON == 16 == config.forecast.horizon,
    "two or three folds": 2 <= len(folds) <= 3,
    "every validation window has 16 dates": all(len(fold.validation_dates) == 16 for fold in folds),
    "every validation grid is complete": all(
        run["validation_rows"] == dataset_summary.series * PROJECT_HORIZON
        and run["validation_series"] == dataset_summary.series
        and run["duplicate_row_keys"] == 0
        for run in fold_runs
    ),
    "preprocessors stop at each origin": all(
        run["preprocessor_fitted_through"] == run["origin"] for run in fold_runs
    ),
    "no transformed missing values": all(run["missing_after_transform"] == 0 for run in fold_runs),
    "split manifest exists": split_path.is_file(),
    "processed panel not exported": True,
    "leakage tests passed": test_result.returncode == 0,
}
contract = model_input_contract(example_batch)
test_output_lines = [line for line in test_result.stdout.splitlines() if line.strip()]
run_evidence = {
    "mode": MODE,
    "source": dataset_summary.source,
    "dataset_summary": dataset_summary.to_dict(),
    "official_kaggle_hashes_verified": MODE == "full",
    "official_input_sha256": EXPECTED_KAGGLE_SHA256 if MODE == "full" else None,
    "runtime_versions": capture_versions(),
    "total_elapsed_seconds": round(time.perf_counter() - RUN_STARTED, 4),
    "process_max_rss_kib": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
    "leakage_test_files": ["tests/test_splits.py", "tests/test_features.py"],
    "leakage_test_summary": test_output_lines[-1] if test_output_lines else None,
    "gate_checks": gate_checks,
    "gate_status": "PASS" if all(gate_checks.values()) else "FAIL",
}
manifest = feature_manifest(spec, split_data["sha256"], fold_runs, example_batch, run_evidence)
feature_path = export_feature_manifest(manifest, METRICS_DIR / "feature_manifest.json")
display(
    {
        "row_key": contract["row_key"],
        "tabular_feature_count": manifest["feature_count"],
        "recurrent_target_shape": contract["recurrent"]["target_shape"],
        "feature_manifest_sha256": manifest["sha256"],
        "processed_dataset_exported": manifest["processed_dataset_exported"],
    }
)
print(f"feature manifest: {feature_path}")

In [ ]:
final_gate_checks = {**gate_checks, "feature manifest exists": feature_path.is_file()}
gate_table = pd.DataFrame(
    [
        {"check": name, "status": "PASS" if passed else "FAIL"}
        for name, passed in final_gate_checks.items()
    ]
)
display(gate_table)
if not all(final_gate_checks.values()):
    raise RuntimeError("G3 FAILED: do not continue to model development.")
if MODE == "full":
    print("G3 FULL PASS: feature and rolling-origin contracts are ready for later model work.")
else:
    print("G3 SMOKE PASS: contracts work; reproduce full mode before model development.")

## Development boundary

A **full-mode PASS** closes the data/feature gate: model experiments may begin using these fixed splits and contracts. A smoke PASS only validates the implementation and still requires full reproduction. Neither result selects the final tree or recurrent family, nor demonstrates predictive improvement; those decisions require validation metrics from the model notebooks.